# YOLOv11s vs YOLOv12s — Comparative Evaluation

Side-by-side comparison of the two architectures evaluated in this dissertation.

| | YOLOv11s | YOLOv12s |
|---|---|---|
| Architecture | CSP + C3k2 + C2PSA | CSP + A2C2f (area-attention) |
| Parameters | 9.41 M | 9.10 M |
| GFLOPs | 21.3 | 19.6 |
| Dataset | Roboflow smoking (5-class) | Roboflow smoking (5-class) |
| Epochs | 50 | 50 |
| Image size | 320x320 | 320x320 |

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'figure.facecolor': '#0e0e1a',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#3a3a5c',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#9ba8b5',
    'ytick.color':      '#9ba8b5',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2d2d42',
    'legend.facecolor': '#1a1a2e',
    'legend.edgecolor': '#3a3a5c',
})
print('Setup complete.')

## 2. Overall Metrics — Roboflow External Test Set

In [ ]:
models = ['YOLOv11s', 'YOLOv12s']
external = {
    'mAP50':     [0.721, 0.706],
    'Precision': [0.848, 0.822],
    'Recall':    [0.686, 0.651],
    'F1':        [0.759, 0.727],
}
print(f"{'Metric':<16} {'YOLOv11s':>10} {'YOLOv12s':>10} {'Winner':>12}")
print('-' * 52)
for k, (v11, v12) in external.items():
    winner = 'YOLOv11s' if v11 >= v12 else 'YOLOv12s'
    print(f'{k:<16} {v11:>10.3f} {v12:>10.3f} {winner:>12}')

## 3. Bar Chart — Overall Metrics

In [ ]:
metrics  = ['mAP@50', 'Precision', 'Recall', 'F1']
v11_vals = [external['mAP50'][0], external['Precision'][0],
            external['Recall'][0], external['F1'][0]]
v12_vals = [external['mAP50'][1], external['Precision'][1],
            external['Recall'][1], external['F1'][1]]
x, w = np.arange(len(metrics)), 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, v11_vals, w, label='YOLOv11s', color='#FF7814', alpha=0.9)
b2 = ax.bar(x + w/2, v12_vals, w, label='YOLOv12s', color='#3250E6', alpha=0.9)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom',
            fontsize=9, color='#e0e0e0')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('YOLOv11s vs YOLOv12s — External Test Metrics', fontsize=13, pad=12)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.legend(); ax.yaxis.grid(True, linestyle='--', alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('../assets/training_results/comparison_external_metrics.png',
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved comparison_external_metrics.png')

## 4. Per-Class mAP@50

In [ ]:
classes  = ['Cigarette', 'Person', 'Smoke', 'Vape', 'Smoking']
v11_cls  = [0.800, 0.930, 0.560, 0.360, 0.880]
v12_cls  = [0.375, 0.734, 0.269, 0.358, 0.448]
x, w = np.arange(len(classes)), 0.35
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, v11_cls, w, label='YOLOv11s', color='#FF7814', alpha=0.9)
ax.bar(x + w/2, v12_cls, w, label='YOLOv12s', color='#3250E6', alpha=0.9)
ax.set_ylim(0, 1.05); ax.set_ylabel('mAP@50')
ax.set_title('Per-Class mAP@50', fontsize=13, pad=12)
ax.set_xticks(x); ax.set_xticklabels(classes)
ax.legend(); ax.yaxis.grid(True, linestyle='--', alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('../assets/training_results/comparison_per_class_map.png',
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 5. Model Efficiency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, vals, ylabel, title in [
    (axes[0], [18.4, 18.2], 'Size (MB)', 'Weights File Size'),
    (axes[1], [21.3, 19.6], 'GFLOPs',    'Computational Cost'),
]:
    bars = ax.bar(models, vals, color=['#FF7814','#3250E6'], alpha=0.9, width=0.4)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                str(v), ha='center', fontsize=11, color='#e0e0e0')
    ax.set_ylabel(ylabel); ax.set_title(title)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5); ax.set_axisbelow(True)
plt.suptitle('Model Efficiency Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('../assets/training_results/comparison_efficiency.png',
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 6. Final Selection Rationale

In [ ]:
rows = [
    ('mAP@50 (external)',  '72.1%',   '70.6%',   'YOLOv11s +1.5pp'),
    ('Precision',          '84.8%',   '82.2%',   'YOLOv11s +2.6pp'),
    ('Recall',             '68.6%',   '65.1%',   'YOLOv11s +3.5pp'),
    ('F1 Score',           '75.9%',   '72.7%',   'YOLOv11s +3.2pp'),
    ('Parameters',         '9.41 M',  '9.10 M',  'YOLOv12s -310K'),
    ('GFLOPs',             '21.3',    '19.6',    'YOLOv12s -1.7'),
    ('Weights size',       '18.4 MB', '18.2 MB', 'Similar'),
]
print(f"  {'Metric':<22} {'YOLOv11s':>10} {'YOLOv12s':>10}  {'Advantage':>20}")
print('  ' + '-'*68)
for r in rows:
    print(f'  {r[0]:<22} {r[1]:>10} {r[2]:>10}  {r[3]:>20}')
print('\nConclusion: YOLOv11s outperforms YOLOv12s across all detection metrics.\n'
      'YOLOv11s selected as the deployment model for this dissertation.')